# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 5: *Feature Interaction Analysis*
##### Version Number: 2.0
---
### Contents  
> 1. *Filter Dates for modeling*
> 2. *Split and Scale Data*
> 3. *Export Data*
---
### Notes
---
### Inputs
- `final.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `X`,`y` - for future modeling
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions
from src.data_utils import create_interactions

# A space saving function to rank interactions
from src.data_utils import rank_variables_by_correlation

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling prep
from sklearn.preprocessing import MinMaxScaler

---

###  Load Data

In [3]:
y_full = pd.read_csv("../data/processed/y.csv")
X_full = pd.read_csv("../data/processed/X.csv")
details_full = pd.read_csv("../data/processed/details.csv")

## Downsize majority class to save processor

In [4]:
final = pd.concat([details_full, X_full, y_full], axis=1)

In [5]:
final['Date'] = pd.to_datetime(final['Date'])

# Number of rows to keep from class 0
n_keep = 50000

# Split by class
class0 = final[final['Target'] == 0]
class1 = final[final['Target'] == 1]
class2 = final[final['Target'] == 2]

# Sample class 0 down to 50,000 rows
class0_sampled = class0.sample(n=n_keep, random_state=14)

# Combine back together
reduced = pd.concat([class0_sampled, class1, class2], ignore_index=True)

In [6]:
reduced['Target'].value_counts()

Target
0    50000
1    43005
2    13773
Name: count, dtype: int64

In [7]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target', 
      'Region_Name','Eco_Name',
       'County', 'Sample_Longitude', 'Sample_Latitude',
       'geometry', 'fire_count', 'total_fire_damage',
        '_merge','acres']

y_reduced = reduced['Target']
X_reduced = reduced.drop(columns=text_columns)
details_reduced = reduced[text_columns]

Scale all datasets and save back to dataframe format. MinMax Scaler used because majority of values are > 0

In [8]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_reduced)
X_scaled = pd.DataFrame(X_scaled, columns=X_reduced.columns, index=X_reduced.index)

In [9]:
rank_variables_by_correlation(X_scaled,y_reduced)

,Feature,Correlation
0,Influence_Area,0.339491
1,Intermix_Area,0.316092
2,2-Year Avg Fires,0.303555
3,Year,0.294765
4,Solar Radiation 7 Day Avg,0.274355
5,Daily Maximum Air Temperature 7 Day Avg,0.257074
6,Solar Radiation,0.256187
7,Daily Maximum Air Temperature,0.255268
8,Daily Minimum Air Temperature 7 Day Avg,0.242610
9,Daily Minimum Air Temperature,0.240541


---

In [10]:
X_scaled = X_scaled.drop(columns='Precipitation')

## 3. Export Data

In [11]:
X_reduced.to_csv('../data/processed/X_scaled.csv', index=False)
y_reduced.to_csv('../data/processed/y_reduced.csv', index=False)
details_reduced.to_csv('../data/processed/details_reduced.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/


In [12]:
X_reduced.columns

Index(['Burning Index', 'Energy Release Component',
       'Actual Evapotranspiration', '100-hour Dead Fuel Moisture',
       '1000-hour Dead Fuel Moisture', 'Precipitation',
       'Maximum Relative Humidity', 'Minimum Relative Humidity',
       'Specific Humidity', 'Solar Radiation', 'Daily Minimum Air Temperature',
       'Daily Maximum Air Temperature', 'Vapor Pressure Deficit', 'Wind Speed',
       'Standardized Precipitation Index 30-Day',
       'Standardized Precipitation Index 180-Day',
       'Standardized Precipitation Evapotranspiration Index 30-Day',
       'Standardized Precipitation Evapotranspiration Index 90-Day',
       'Standardized Precipitation Evapotranspiration Index 180-Day',
       'Palmer Drought Severity Index', 'Sample_Elevation', 'Region_ID',
       'Interface_Area', 'Intermix_Area', 'Influence_Area', 'Total_Housing',
       'Total_Population', 'Eco_ID', 'Buffered_Area_Km', 'Population_Density',
       'Housing_Density', 'Mean Income', 'Season', 'Month', 'Y